In [ ]:
"""
Wikipedia Plot Scraper
Section 1.2 of the Master Engineering Specification

EXACT IMPLEMENTATION:
- Uses wikipedia-api library (not raw HTML)
- Query format: "{movie_title} film {release_year}"
- Uses opensearch to resolve exact page titles
- Fetches extract field from summary endpoint
- 0.5 second rate limiting
- Caching: never rescrape existing entries
"""

import time
import json
import requests
import pandas as pd
from pathlib import Path
from typing import Dict, Optional

# ============================================================================
# CONFIGURATION - EXACTLY AS SPECIFIED
# ============================================================================
USER_AGENT = "MovieRecommenderBot/1.0 (your-email@example.com)"
SLEEP_TIME = 0.5  # EXACT: "add a 0.5 second sleep between requests"
OUTPUT_PATH = Path("../data/external/wikipedia_plots.json")
MOVIES_PATH = Path("../data/processed/movies_merged.csv")

# Wikipedia API endpoints (as specified in Section 1.2)
OPENSEARCH_URL = "https://en.wikipedia.org/w/api.php"
SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"

# Create output directory if it doesn't exist
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================================
# STEP 1: LOAD EXISTING CACHE (NEVER RESCRAPE)
# ============================================================================
def load_existing_cache() -> Dict[str, str]:
    """Load existing Wikipedia plots cache if it exists."""
    if OUTPUT_PATH.exists():
        with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}


# ============================================================================
# STEP 2: RESOLVE PAGE TITLE USING OPENSEARCH (EXACTLY AS SPECIFIED)
# ============================================================================
def resolve_page_title(movie_title: str, release_year: int) -> Optional[str]:
    """
    Use Wikipedia opensearch endpoint to resolve exact page title.
    Query format: "{movie_title} film {release_year}"
    
    Args:
        movie_title: Title of the movie
        release_year: Release year for disambiguation
    
    Returns:
        Resolved Wikipedia page title or None if not found
    """
    query = f"{movie_title} film {release_year}"
    
    params = {
        "action": "opensearch",
        "search": query,
        "limit": 1,
        "namespace": 0,
        "format": "json"
    }
    
    headers = {"User-Agent": USER_AGENT}
    
    try:
        response = requests.get(OPENSEARCH_URL, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        # opensearch returns [query, [titles], [descriptions], [urls]]
        titles = data[1]
        return titles[0] if titles else None
        
    except Exception as e:
        print(f"  Opensearch error for '{query}': {str(e)}")
        return None


# ============================================================================
# STEP 3: FETCH EXTRACT FROM SUMMARY ENDPOINT (EXACTLY AS SPECIFIED)
# ============================================================================
def fetch_plot_summary(page_title: str) -> Optional[str]:
    """
    Fetch the extract field from the Wikipedia summary endpoint.
    This contains the first several paragraphs, always including the plot.
    
    Args:
        page_title: Exact Wikipedia page title
    
    Returns:
        Plot text or None if fetch fails
    """
    url = SUMMARY_URL.format(title=page_title.replace(" ", "_"))
    headers = {"User-Agent": USER_AGENT}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # EXACTLY as specified: "fetch the extract field"
        return data.get("extract", "").strip()
        
    except Exception as e:
        print(f"  Summary fetch error for '{page_title}': {str(e)}")
        return None


# ============================================================================
# STEP 4: MAIN SCRAPING LOOP
# ============================================================================
def main():
    print("=" * 60)
    print("WIKIPEDIA PLOT SCRAPER - Section 1.2")
    print("=" * 60)
    
    # Load movies that need scraping
    print(f"\nLoading movies from: {MOVIES_PATH}")
    movies_df = pd.read_csv(MOVIES_PATH)
    
    # We only need ID, title, and release_year
    movies_df = movies_df[['id', 'title', 'release_year']].dropna()
    movies_df['release_year'] = movies_df['release_year'].astype(int)
    
    print(f"Total movies in dataset: {len(movies_df)}")
    
    # Load existing cache
    wiki_plots = load_existing_cache()
    print(f"Movies already cached: {len(wiki_plots)}")
    
    # Filter to movies not yet cached
    movies_to_scrape = movies_df[~movies_df['id'].astype(str).isin(wiki_plots.keys())]
    print(f"Movies to scrape: {len(movies_to_scrape)}")
    
    if len(movies_to_scrape) == 0:
        print("\n✓ No new movies to scrape. Cache is up to date.")
        return
    
    # Scraping statistics
    successful = 0
    failed = 0
    failures_log = []
    
    print("\n" + "=" * 60)
    print("STARTING SCRAPE")
    print("=" * 60)
    
    for idx, row in movies_to_scrape.iterrows():
        tmdb_id = str(row['id'])
        title = row['title']
        year = row['release_year']
        
        print(f"\n[{idx+1}/{len(movies_df)}] Processing: {title} ({year}) [TMDB: {tmdb_id}]")
        
        # Step 1: Resolve page title using opensearch
        page_title = resolve_page_title(title, year)
        if not page_title:
            print(f"  ✗ Could not resolve page title")
            wiki_plots[tmdb_id] = ""
            failed += 1
            failures_log.append((tmdb_id, title, year, "No page title resolved"))
            time.sleep(SLEEP_TIME)
            continue
            
        print(f"  ✓ Resolved page: {page_title}")
        
        # Step 2: Fetch plot from summary endpoint
        plot_text = fetch_plot_summary(page_title)
        if not plot_text:
            print(f"  ✗ Could not fetch plot")
            wiki_plots[tmdb_id] = ""
            failed += 1
            failures_log.append((tmdb_id, title, year, "No plot text fetched"))
            time.sleep(SLEEP_TIME)
            continue
        
        # Success!
        wiki_plots[tmdb_id] = plot_text
        successful += 1
        print(f"  ✓ Plot fetched ({len(plot_text)} chars)")
        
        # Rate limiting - EXACTLY 0.5 seconds as specified
        print(f"  Sleeping {SLEEP_TIME}s...")
        time.sleep(SLEEP_TIME)
        
        # Save checkpoint every 50 movies (optional, not in spec but safe)
        if (successful + failed) % 50 == 0:
            with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
                json.dump(wiki_plots, f, ensure_ascii=False, indent=2)
            print(f"\n  CHECKPOINT: Saved {len(wiki_plots)} entries to cache")
    
    # ============================================================================
    # STEP 5: FINAL SAVE
    # ============================================================================
    print("\n" + "=" * 60)
    print("SCRAPING COMPLETE")
    print("=" * 60)
    
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(wiki_plots, f, ensure_ascii=False, indent=2)
    
    print(f"\nFinal cache file: {OUTPUT_PATH}")
    print(f"Total entries: {len(wiki_plots)}")
    print(f"Successfully fetched: {successful}")
    print(f"Failed: {failed}")
    
    if failures_log:
        print("\nSample failures (first 10):")
        for fail in failures_log[:10]:
            print(f"  - {fail[1]} ({fail[2]}): {fail[3]}")
    
    # Expected coverage: "Expect approximately 30–40% of movies to either not have a dedicated Wikipedia page"
    success_rate = (successful / len(movies_to_scrape)) * 100 if len(movies_to_scrape) > 0 else 0
    print(f"\nSuccess rate for new movies: {success_rate:.1f}%")
    print(f"(Spec expects 60-70% success rate)")


if __name__ == "__main__":
    main()

C:\Users\ADEGOKE\AppData\Local\Temp\ipykernel_1624\3256927813.py:13: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('../data/processed/movies_merged.csv')  # or from DB


[1/45338] Querying: Toy Story film 1995
[2/45338] Querying: Jumanji film 1995
[3/45338] Querying: Grumpier Old Men film 1995
[4/45338] Querying: Waiting to Exhale film 1995
[5/45338] Querying: Father of the Bride Part II film 1995
[6/45338] Querying: Heat film 1995
[7/45338] Querying: Sabrina film 1995
[8/45338] Querying: Tom and Huck film 1995
[9/45338] Querying: Sudden Death film 1995
[10/45338] Querying: GoldenEye film 1995
[11/45338] Querying: The American President film 1995
[12/45338] Querying: Dracula: Dead and Loving It film 1995
[13/45338] Querying: Balto film 1995
[14/45338] Querying: Nixon film 1995
[15/45338] Querying: Cutthroat Island film 1995
[16/45338] Querying: Casino film 1995
[17/45338] Querying: Sense and Sensibility film 1995
[18/45338] Querying: Four Rooms film 1995
[19/45338] Querying: Ace Ventura: When Nature Calls film 1995
[20/45338] Querying: Money Train film 1995
[21/45338] Querying: Get Shorty film 1995
[22/45338] Querying: Copycat film 1995
[23/45338] Quer

KeyboardInterrupt: 